# Ουδέτερες ετικέτες: μετράει η λέξη ή το παίγνιο;

**Η ένσταση.** Σε κάθε run ως τώρα οι δύο πράξεις λέγονται «Cooperate» και
«Defect», λέξεις με ηθικό πρόσημο. Αν η συνεργασία κρατιέται επειδή η λέξη
είναι «καλή», ή αν η ανταγωνιστική βλάβη έρχεται επειδή τα μηνύματα γράφουν
«Defect», τότε μέρος των ευρημάτων αφορά το λεξιλόγιο και όχι τη στρατηγική.

**Το πείραμα.** Ίδιο παίγνιο, ίδιες αποδόσεις, ίδια σειρά στον πίνακα· οι δύο
πράξεις λέγονται **J** και **F**. Αλλάζουν μόνο οι λέξεις που διαβάζει και
γράφει ο πράκτορας: ο πίνακας αποδόσεων, το «Actions available», η μορφή της
απάντησης και το ιστορικό. Το record κρατά τις ετικέτες του παιγνίου, οπότε
όλη η ανάλυση μένει ίδια. Μια απάντηση που δεν είναι J ή F, ακόμα και το
«Cooperate», γράφεται άκυρη· το ακατέργαστο κείμενο μένει στο `llm_calls`.

| κελί | τι ελέγχει |
|---|---|
| `no_comm` | αν η λέξη κινεί τη διάθεση χωρίς κανάλι |
| `baseline_cheap_talk` | αν το RQ1 — η κουβέντα ανεβάζει τη συνεργασία — αντέχει χωρίς τις λέξεις |
| `framing_competitive` | αν η ανταγωνιστική βλάβη χρειάζεται τη λέξη «Defect» |

Αστέρας, PD, n=5, με το `max_tokens` του πλέγματος (session A για το baseline,
B για το ανταγωνιστικό). Στην ανάλυση τα κελιά λέγονται `no_comm+neutral`,
`baseline_cheap_talk+neutral` και `framing_competitive+neutral`. Το tag μπαίνει
και στα δύο σκέλη, γιατί οι λέξεις φτάνουν και στο κλειστό κανάλι, και τα Δ των
ουδέτερων κελιών μετριούνται απέναντι στο `no_comm+neutral`, ποτέ στο κανονικό.

### Κανόνες απόφασης, γραμμένοι πριν τρέξει οτιδήποτε (5 Οκτωβρίου 2026)

Άγκυρες με τις κανονικές ετικέτες, αστέρας, PD, μέσος όρος run:

| μοντέλο | `no_comm` | `baseline_cheap_talk` | `framing_competitive` (n=10) |
|---|---|---|---|
| Qwen2.5-7B | 0,08 | 1,00 | 0,18 |
| Llama-3.1-8B | 0,47 | 0,98 | 0,32 |
| Qwen3-4B (προαιρετικό) | 0,04 | 1,00 | 0,01 |

1. **Το RQ1 χωρίς τις λέξεις.** Κρατά αν με J/F το `baseline_cheap_talk` είναι
   ≥ 0,90 και ξεπερνά το `no_comm` κατά ≥ 0,30, σε κάθε μοντέλο. Αν πέσει κάτω
   από 0,70 σε κάποιο μοντέλο, για εκείνο το μοντέλο το RQ1 γράφεται ως
   εξαρτώμενο από τις ετικέτες.
2. **Η διάθεση (`no_comm`).** Αν η διαφορά J/F − κανονικό είναι ≥ 0,30 σε
   απόλυτη τιμή, οι λέξεις κινούν τη διάθεση και αναφέρεται η κατεύθυνση. Αν
   είναι μικρότερη, είναι μέσα στον θόρυβο του n=5 (το A/A του αστέρα έδωσε
   έως ±0,29) και δεν είναι ένδειξη.
3. **Η ανταγωνιστική βλάβη.** Αν με J/F το `framing_competitive` μένει ≤ 0,50
   σε κάθε μοντέλο, η βλάβη δεν χρειάζεται τη λέξη «Defect». Αν ανέβει ≥ 0,80
   σε κάποιο μοντέλο, εκεί η βλάβη ήταν εν μέρει λεξιλογική.
4. Κελί με περισσότερες από 10% άκυρες απαντήσεις αναφέρεται, αλλά δεν μετρά
   στους κανόνες. Αναφέρεται επίσης πόσες άκυρες απαντήσεις ονομάζουν τη λέξη
   του παιγνίου («Cooperate», «Defect»): ένα άμεσο μέτρο του πόσο φέρνει το
   μοντέλο μόνο του την ιδέα του διλήμματος.

## Setup

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN` (για το Llama)

Μετά **Save & Run All**, αλλάζοντας μόνο το `STEP`. Κάθε βήμα είναι ένα
μοντέλο, οπότε με περισσότερους λογαριασμούς τα βήματα τρέχουν παράλληλα.

| βήμα | μοντέλο | κελιά | runs | ~ώρες |
|---|---|---|---|---|
| 1 | Qwen2.5-7B | no_comm, baseline_cheap_talk, framing_competitive | 15 | 2 |
| 2 | Llama-3.1-8B | τα ίδια | 15 | 3,5 |
| 3 | Qwen3-4B (προαιρετικό) | τα ίδια | 15 | 4 |

Οι ώρες βγαίνουν από μετρημένους χρόνους ανά run: Qwen2.5 ~9 λεπτά με ανοιχτό
κανάλι, Llama ~16, Qwen3-4B ~18, και περίπου τα μισά με κλειστό. Ο Qwen3-4B
είναι το μοντέλο που γράφει κυριολεκτικά «Defect» στα ανταγωνιστικά του
μηνύματα, άρα ο πιο καθαρός έλεγχος για το αν η βλάβη είναι λεξιλογική.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU off -- Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

for f in ('campaign.py', 'run_all_scenarios.py'):
    assert '--action-labels' in open(f, encoding='utf-8').read(), (
        f'Το {f} δεν έχει --action-labels -- κάνε git push πρώτα.')

print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

## Ο έλεγχος χωρίς GPU

Δύο γύροι με ψεύτικο μοντέλο, με κλειστό και με ανοιχτό κανάλι. Διαβάζει τα ίδια
τα prompts: ο πράκτορας πρέπει να βλέπει παντού J/F και πουθενά
«Cooperate/Defect». Το record πρέπει να κρατά τις ετικέτες του παιγνίου, και η
αντιστοίχιση πρέπει να δέχεται το «J» και το «Option J» αλλά όχι το «just» ή το
«Cooperate».

In [ ]:
import re, sys
sys.path.insert(0, '.')
from agent import Agent
from config import ExperimentConfig, ModelConfig
from engine import make_engine
from games import GAMES

OWN = re.compile(r'\b(Cooperate|Defect)\b')

class Echo:
    """Στη θέση του LLM: J στον γύρο 1, F στον γύρο 2· κρατά κάθε prompt."""
    last_meta = {}
    def __init__(self):
        self.prompts = []
    def generate(self, system, user):
        self.prompts.append((system, user))
        if 'COMMUNICATION PHASE' in user:
            return '{"reasoning": "-", "message": "let us both pick J"}'
        return '{"reasoning": "-", "action": "%s"}' % ('J' if re.search(r'Current round: 1\b', user) else 'F')

for condition in ('no_comm', 'cheap_talk'):
    cfg = ExperimentConfig(game='pd', condition=condition, topology='star', n_rounds=2,
                           model=ModelConfig(provider='local'), action_labels='neutral')
    engine, echo = make_engine(cfg), Echo()
    rec = engine.run_one(run_id=1, client=echo)
    systems = {s for s, _ in echo.prompts}
    chose = [l for _, u in echo.prompts for l in u.splitlines() if ' chose: ' in l]
    print(f'{condition:10s} {len(echo.prompts):2d} prompts | record: '
          f'{[r["actions"][0] for r in rec["history"]]} | ιστορικό: {sorted(set(l.split(": ")[1][:1] for l in chose))}')
    assert all('Actions available: J or F.' in s and '"action": "<J or F>"' in s for s in systems), \
        'λάθος προσφερόμενες πράξεις -- σταμάτα'
    assert not any(OWN.search(s) for s in systems), 'το system prompt ονομάζει Cooperate/Defect -- σταμάτα'
    assert chose and not any(OWN.search(l) for l in chose), 'το ιστορικό ονομάζει Cooperate/Defect -- σταμάτα'
    assert [r['actions'][0] for r in rec['history']] == ['Cooperate', 'Defect'], \
        'το record πρέπει να κρατά τις ετικέτες του παιγνίου (J -> Cooperate, F -> Defect) -- σταμάτα'

ag = Agent(agent_id=0, n_neighbors=1, neighbor_ids=[1], total_agents=4, game=GAMES['pd'],
           condition='no_comm', client=None, action_labels='neutral')
cases = {'J': 'Cooperate', 'j.': 'Cooperate', 'Option J': 'Cooperate', 'F (defect)': 'Defect',
         'F': 'Defect', 'Cooperate': '', 'just': '', 'final': '', '': ''}
for raw, want in cases.items():
    assert ag._canonicalize_action(raw) == want, (raw, ag._canonicalize_action(raw), want)
print(f'αντιστοίχιση: {len(cases)} περιπτώσεις σωστές· «Cooperate», «just», «final» γράφονται άκυρα')

pd_game = GAMES['pd']
assert re.findall(r'\d', pd_game.describe_payoffs()) == re.findall(r'\d', pd_game.describe_payoffs(('J', 'F')))
print('ίδιες αποδόσεις, μόνο τα ονόματα αλλάζουν:\n' + pd_game.describe_payoffs(('J', 'F')))
print('\nόλα εντάξει')

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    print('Χωρίς HF_TOKEN (εντάξει μόνο για Qwen):', e)

## Ποιο βήμα

In [ ]:
STEP = 1          # <-- ΜΟΝΟ ΑΥΤΟ ΑΛΛΑΖΕΙΣ: 1 Qwen2.5, 2 Llama, 3 Qwen3-4B (προαιρετικό)

Q25, LLA, Q3 = 'Qwen/Qwen2.5-7B-Instruct', 'meta-llama/Llama-3.1-8B-Instruct', 'Qwen/Qwen3-4B'
STEPS = {1: [Q25], 2: [LLA], 3: [Q3]}
MODELS = STEPS[STEP]
NEEDS_HF = any(m.startswith(('meta-llama/', 'google/')) for m in MODELS)

def calls_for(model):
    """(όνομα, ορίσματα, run files που περιμένουμε), με σειρά προτεραιότητας."""
    short = model.split('/')[-1]
    common = ['--model', model, '--topology', 'star', '--games', 'pd', '--action-labels', 'neutral']
    return [
        # Το baseline τρέχει και τα δύο σκέλη: no_comm και cheap_talk.
        (f'{short} · baseline (no_comm + cheap_talk)',
         common + ['--session', 'A', '--scenarios', 'baseline'], 10),
        (f'{short} · framing_competitive',
         common + ['--session', 'B', '--scenarios', 'framing_competitive'], 5),
    ]

CALLS = [c for m in MODELS for c in calls_for(m)]
print(f'βήμα {STEP}: {len(CALLS)} κλήσεις')
for name, _, n in CALLS:
    print(f'   {name}  -> {n} runs')

## Preview — κοστίζει μηδέν

Σε κάθε κλήση πρέπει να δεις `topology : star`, το σωστό session, τα run files
που περιμένουμε (10 για το baseline, 5 για το ανταγωνιστικό) και τη γραμμή
`LABELS : neutral`. Καμία άλλη γραμμή διακόπτη (`FILTER`, `OWN MESSAGE`,
`WRITERS`, `ARMS`): εδώ αλλάζουν μόνο οι ετικέτες.

In [ ]:
import os, subprocess, sys

ok = True
if NEEDS_HF and not os.environ.get('HUGGINGFACE_API_KEY'):
    ok = False
    print('ΛΕΙΠΕΙ ΤΟ HF_TOKEN -- το μοντέλο είναι gated, θα αποτύχει σε ~1 λεπτό.')
for name, extra, expected in CALLS:
    print('=' * 70)
    r = subprocess.run([sys.executable, 'campaign.py', *extra, '--dry-run'],
                       capture_output=True, text=True)
    out = r.stdout.strip()
    print(out[:900])
    if r.returncode != 0:
        ok = False
        print('ΣΦΑΛΜΑ:', (r.stderr or r.stdout).strip()[-300:])
    session = extra[extra.index('--session') + 1]
    for needle in ('topology     : star', f'expecting    : {expected} run files',
                   f'session {session}', 'LABELS       : neutral', '_neutral'):
        if needle not in out:
            ok = False
            print(f'ΛΕΙΠΕΙ ΑΠΟ ΤΟ ΠΛΑΝΟ: {needle!r}')
    # Μόνο οι ετικέτες αλλάζουν: κανένας άλλος διακόπτης στο πλάνο.
    for line in ('FILTER', 'OWN MESSAGE', 'WRITERS', 'ARMS', 'PROMPT'):
        if '\n' + line in '\n' + out:
            ok = False
            print(f'ΑΠΡΟΣΔΟΚΗΤΟΣ ΔΙΑΚΟΠΤΗΣ στο πλάνο: {line}')

assert ok, 'Κάποιο plan απέτυχε -- μη συνεχίσεις.'
print('\n' + '=' * 70)
print('όλα τα πλάνα εντάξει')

## Εκτέλεση

In [ ]:
import subprocess, sys, time

t0 = time.time()
results = []
for name, extra, expected in CALLS:
    print('\n' + '=' * 70)
    print(f'βήμα {STEP} · {name}   ({(time.time() - t0) / 3600:.1f} h μέχρι τώρα)')
    print('=' * 70, flush=True)
    s = time.time()
    r = subprocess.run([sys.executable, 'campaign.py', *extra])
    mins = (time.time() - s) / 60
    status = 'OK' if r.returncode == 0 else f'FAILED (exit {r.returncode})'
    results.append((name, status, mins))
    print(f'\n--> {name}: {status}, {mins:.0f} λεπτά', flush=True)

print('\n' + '=' * 70)
print('ΑΠΟΛΟΓΙΣΜΟΣ')
for name, s, mins in results:
    print(f'  {s:22s} {mins:6.0f} λ   {name}')
print(f'\nσύνολο {(time.time() - t0) / 3600:.1f} h')

## Έλεγχος των δεδομένων και πρώτη ματιά

Για κάθε κελί: μέσος όρος run, συνεργασία στον γύρο 1 (κενό ιστορικό), ποσοστό
άκυρων απαντήσεων, και πόσες άκυρες απαντήσεις ονόμαζαν τη λέξη του παιγνίου.
Επιβεβαιώνει επίσης ότι κάθε record δηλώνει ουδέτερες ετικέτες και κρατά τις
ετικέτες του παιγνίου.

In [ ]:
import glob, json, os, re, sys, statistics
from collections import defaultdict
sys.path.insert(0, '.')
from analysis import summarise_run

ACTION_FIELD = re.compile(r'"action"\s*:\s*"([^"]*)"', re.I)
GAME_WORD = re.compile(r'\b(cooperat\w*|defect\w*)\b', re.I)

print(f"{'φάκελος / κελί':64s} {'n':>3s} {'coop':>6s} {'γύρος 1':>8s} {'άκυρα':>6s} {'λέξη παιγνίου':>14s}")
for d in sorted(glob.glob('/kaggle/working/results/*_neutral')):
    cells = defaultdict(lambda: {'coop': [], 'first': [0, 0], 'bad': [0, 0], 'word': 0})
    for p in glob.glob(f'{d}/**/*.json', recursive=True):
        if os.path.basename(p).startswith('_') or f'{os.sep}zips{os.sep}' in p:
            continue
        rec = json.load(open(p, encoding='utf-8'))
        cfg = rec['config']
        assert cfg.get('action_labels') == 'neutral', f'όχι ουδέτερες ετικέτες: {p}'
        c = cells[f"{cfg['scenario']}[{cfg['condition']}]"]
        c['coop'].append(summarise_run(rec)['coop_rate_overall'])
        for k, x in enumerate(rec['history']):
            for agent, a in x['actions'].items():
                assert a in ('Cooperate', 'Defect', ''), \
                    f'το record έχει {a!r} -- πρέπει να κρατά τις ετικέτες του παιγνίου: {p}'
                c['bad'][0] += a == ''
                c['bad'][1] += 1
                if k == 0 and a:
                    c['first'][0] += a == 'Cooperate'
                    c['first'][1] += 1
                if a == '':
                    # Τι έγραψε το μοντέλο στη θέση του J/F: αν ονομάζει τη λέξη
                    # του παιγνίου, την έφερε μόνο του.
                    raw = [call.get('raw', '') for call in x.get('llm_calls', {}).get(agent, [])
                           if call.get('phase', '').startswith('action')]
                    said = ACTION_FIELD.search(raw[-1]) if raw else None
                    if said and GAME_WORD.search(said.group(1)):
                        c['word'] += 1
    for name, c in sorted(cells.items()):
        r1 = f"{c['first'][0] / c['first'][1]:.2f}" if c['first'][1] else '--'
        print(f"{os.path.basename(d) + ' / ' + name:64s} {len(c['coop']):3d} "
              f"{statistics.fmean(c['coop']):6.3f} {r1:>8s} {c['bad'][0] / c['bad'][1]:6.1%} {c['word']:14d}")

print('\nάγκυρες με τις κανονικές ετικέτες (αστέρας PD): no_comm / baseline_cheap_talk / framing_competitive')
print('  Qwen2.5 0.08 / 1.00 / 0.18     Llama 0.47 / 0.98 / 0.32     Qwen3-4B 0.04 / 1.00 / 0.01')
print('κανόνες απόφασης: στην αρχή του notebook')

## Μάζεμα

In [ ]:
import glob, os, shutil

zips = sorted(glob.glob('/kaggle/working/*_neutral*.zip'))
print(f'{len(zips)} zip:')
for z in zips:
    print(f'   {os.path.getsize(z)/1e6:6.1f} MB  {os.path.basename(z)}')
if zips:
    box = '/kaggle/working/neutral_labels'
    os.makedirs(box, exist_ok=True)
    for z in zips:
        shutil.copy(z, box)
    tag = '_'.join(m.split('/')[-1] for m in MODELS)
    out = shutil.make_archive(f'/kaggle/working/neutral_labels_step{STEP}_{tag}', 'zip', box)
    print(f'\nκατέβασε αυτό: {out}  ({os.path.getsize(out)/1e6:.1f} MB)')
else:
    print('\nΚΑΝΕΝΑ ZIP -- δες τον απολογισμό παραπάνω.')

## Πώς διαβάζεται

Με τους κανόνες της αρχής του notebook, και μόνο με αυτούς. Το κελί ελέγχου
πιο πάνω δίνει όσα χρειάζονται: μέσο όρο run, γύρο 1, άκυρα και πόσα από αυτά
ονόμαζαν τη λέξη του παιγνίου.

Το zip του session B περιέχει και τα runs του session A, επειδή γράφουν στον
ίδιο φάκελο. Αποσυμπίεσε και τα δύο στον ίδιο φάκελο,
`diplomatikh/neutral_labels/`, και τρέξε `audit_corpus.py` και
`verify_prompts.py`. Το δεύτερο ελέγχει πλέον και ότι κανένα «Cooperate/Defect»
δεν έφτασε στον πράκτορα μέσα από το κείμενο του πρωτοκόλλου.

Η ανάλυση κρατά τα ουδέτερα κελιά χωριστά από τα κανονικά (`+neutral`) ακόμα κι
αν μπουν στο ίδιο `--roots`, και τα Δ τους μετριούνται απέναντι στο
`no_comm+neutral`. Η σύγκριση κανονικού με ουδέτερο κελί γίνεται ρητά, κελί
προς κελί, με τους κανόνες της αρχής.